In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
%matplotlib inline

In [2]:
words = open('../../../src/zero_to_hero/part_02_makemore/names.txt', 'r').read().splitlines()

In [3]:
from itertools import product
def build_stoi(gram: int, vocab=list('.abcdefghijklmnopqrstuvwxyz')):
    key_length = gram-1
    d = {}
    idx = 0
    for ctx in product(vocab, repeat=key_length):
        # '.' can only appear as a prefix, not after a non-'.' character
        dot_positions = [i for i, c in enumerate(ctx) if c == '.']
        if dot_positions and dot_positions[-1] != len(dot_positions) - 1:
            continue  # '.' appears after a non-'.' character — invalid

        if len(ctx) == 1:
            ctx = ctx[0]
        d[ctx] = idx
        idx += 1

    return d

In [4]:
# We had S/E special tokens, but now just "."
chars = sorted(list(set(''.join(words))))
stoi = build_stoi(3)
itos = {i:s for s,i in stoi.items()}

In [5]:
stoi2 = build_stoi(2)
itos2 = {i:s for s,i in stoi2.items()}

In [6]:
# using torch.tensor, compute bigram
special_char = 1
total_chars = 26 + special_char
trigram = torch.zeros((len(stoi), total_chars), dtype=torch.int32)

In [7]:
for w in words:
    chs = [".", "."] + list(w) + ["."]
    #print(chs)
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        idx = stoi[(ch1, ch2)]
        trigram[idx][stoi2[ch3]] += 1

In [8]:
p = trigram[0].float()
p = p/ p.sum()
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [9]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
itos2[ix]

'c'

In [10]:
g = torch.Generator().manual_seed(2147483647)
ix1 = 0
word = ""
it = 0
smoothing = 1
# Model smoothing - see below
P = (trigram + smoothing).float()
P /= P.sum(axis=1, keepdim=True)
while True:
    ix = torch.multinomial(P[ix1], num_samples=1, replacement=True, generator=g).item()
    if ix == 0:
        break
    word += itos2[ix]
    new_context = tuple(itos[ix1][1] + itos2[ix])
    ix1 = stoi[new_context]
    it += 1

print(word)

ce


In [11]:
# GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling) i.e. the bigram counts
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood
# equivalent to minimizing the average negative log likelihood

# log(a*b*c) = log(a) + log(b) + log(c)

In [12]:
log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.', '.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    idx = stoi[(ch1, ch2)]
    idx2 = stoi2[ch3]
    prob = P[idx, idx2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-504653.)
nll=tensor(504653.)
2.2119739055633545


In [13]:
# Now lets train to get the best NLL
# NN is going to go character -> probability of what comes next
# create training set

In [14]:
# create training set
xs, ys = [], []

for w in words[:1]:
  chs = ['.', '.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    idx = stoi[(ch1, ch2)]
    idx2 = stoi2[ch3]
    xs.append(idx)
    ys.append(idx2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [15]:
xs.dtype

torch.int64

In [16]:
# What should input be? Integers doesn't make sense (why would you multiply by an index??)
# We'll use one_hot encoding for now
# torch.nn.functional.one_hot

In [17]:
xs

tensor([  0,   5, 143, 351, 339])

In [18]:
import torch.nn.functional as F
# dtype preserved from xs unless otherwise (?)
xenc = F.one_hot(xs, num_classes=len(stoi)).float()

In [19]:
xenc.dtype

torch.float32

In [20]:
xenc.shape

torch.Size([5, 703])

In [21]:
# randn - tensor of shape with std normal
# for each of 27 neurons, what is firing rate on the qth input
# (xenc @ W)[3,13] - 13th neuron, 3rd example
W = torch.randn((703,27))
(xenc @ W).shape

torch.Size([5, 27])

In [22]:
# trying to produce probability definition - softmax
# interpret outputs as log counts - exponentiate them, then do probability normalisation - same things

In [23]:
logits = xenc @ W # log-counts, recall "logit"
counts = logits.exp() # equivalent to trigram
probs = counts / counts.sum(1, keepdims=True)
probs

tensor([[0.0034, 0.0551, 0.0030, 0.0099, 0.0110, 0.0586, 0.0766, 0.1773, 0.0654,
         0.0384, 0.0095, 0.0168, 0.1693, 0.0140, 0.0167, 0.0053, 0.0162, 0.0184,
         0.0277, 0.0112, 0.0269, 0.0141, 0.0169, 0.0249, 0.0056, 0.1028, 0.0049],
        [0.1117, 0.0962, 0.0091, 0.0103, 0.0057, 0.0070, 0.0270, 0.0560, 0.0067,
         0.0197, 0.0053, 0.0055, 0.0123, 0.0121, 0.0079, 0.1598, 0.0568, 0.0034,
         0.0305, 0.0297, 0.0451, 0.1981, 0.0123, 0.0285, 0.0088, 0.0260, 0.0083],
        [0.0246, 0.0114, 0.0786, 0.0333, 0.0056, 0.0699, 0.0481, 0.0431, 0.0194,
         0.0621, 0.0041, 0.0335, 0.0247, 0.0271, 0.0069, 0.0051, 0.0140, 0.0461,
         0.0886, 0.0633, 0.0196, 0.0198, 0.0357, 0.0894, 0.0805, 0.0320, 0.0134],
        [0.0795, 0.0351, 0.0183, 0.0683, 0.0558, 0.0212, 0.0135, 0.0195, 0.0489,
         0.0596, 0.0258, 0.0364, 0.0168, 0.0024, 0.0177, 0.1064, 0.0582, 0.0504,
         0.0047, 0.0345, 0.0044, 0.1057, 0.0371, 0.0264, 0.0198, 0.0199, 0.0138],
        [0.1999, 0.0232,

In [24]:
# Summary
n_inp = len(stoi)

In [25]:
# randomly initialize 27 neurons' weights. each neuron receives 703 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((n_inp, total_chars), generator=g)

In [26]:
xenc = F.one_hot(xs, num_classes=n_inp).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# btw: the last 2 lines here are together called a 'softmax' - numerically stable. Gumbel

In [27]:
probs.shape

torch.Size([5, 27])

In [28]:
n_trigram = probs.shape[0]
nlls = torch.zeros(n_trigram)
for i in range(n_trigram):
  # i-th bigram:
  x = xs[i].item() # input character index
  y = ys[i].item() # label character index
  print('--------')
  print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

--------
bigram example 1: ('.', '.')('.', 'e') (indexes 0,5)
input to the neural net: 0
output probabilities from the neural net: tensor([0.0607, 0.0100, 0.0123, 0.0042, 0.0168, 0.0123, 0.0027, 0.0232, 0.0137,
        0.0313, 0.0079, 0.0278, 0.0091, 0.0082, 0.0500, 0.2378, 0.0603, 0.0025,
        0.0249, 0.0055, 0.0339, 0.0109, 0.0029, 0.0198, 0.0118, 0.1537, 0.1459])
label (actual next character): 5
probability assigned by the net to the the correct character: 0.012286250479519367
log likelihood: -4.3992743492126465
negative log likelihood: 4.3992743492126465
--------
bigram example 2: ('.', 'e')('.', 'm') (indexes 5,13)
input to the neural net: 5
output probabilities from the neural net: tensor([0.0290, 0.0796, 0.0248, 0.0521, 0.1989, 0.0289, 0.0094, 0.0335, 0.0097,
        0.0301, 0.0702, 0.0228, 0.0115, 0.0181, 0.0108, 0.0315, 0.0291, 0.0045,
        0.0916, 0.0215, 0.0486, 0.0300, 0.0501, 0.0027, 0.0118, 0.0022, 0.0472])
label (actual next character): 13
probability assigned by t

In [29]:
# --------- !!! OPTIMIZATION STEP !!! --------------

In [30]:
# randomly initialize 27 neurons' weights. each neuron receives 703 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((n_inp, total_chars), generator=g, requires_grad=True)

In [31]:
# forward pass
xenc = F.one_hot(xs, num_classes=n_inp).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
# See you get this
loss = -probs[torch.arange(n_trigram), ys].log().mean()

In [32]:
print(loss.item())

3.435041904449463


In [33]:
# backward pass
W.grad = None # set to zero the gradient (why None and not zero? Because W is a tensor, seems to be standard)
loss.backward()

In [34]:
W.data += -0.1 * W.grad

In [35]:
# --------- !!! OPTIMIZATION !!! but this time actually --------------

In [36]:
# create the dataset
xs, ys = [], []
for w in words:
  chs = ['.', '.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    idx = stoi[(ch1, ch2)]
    idx2 = stoi2[ch3]
    xs.append(idx)
    ys.append(idx2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network' - note that weights need grad
g = torch.Generator().manual_seed(2147483647)
# W all 0 is like max smoothing from above
W = torch.randn((n_inp, total_chars), generator=g, requires_grad=True)

number of examples:  228146


In [37]:
# gradient descent
for k in range(1000):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=n_inp).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts - just takes rows of W. See the connection to the count procedure!
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  # Note we regularise below
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

3.8183300495147705
3.663242816925049
3.5703790187835693
3.5015382766723633
3.444596290588379
3.394578695297241
3.3496899604797363
3.3089394569396973
3.271679639816284
3.2374250888824463
3.2057759761810303
3.1763992309570312
3.149017095565796
3.123403549194336
3.0993728637695312
3.076773166656494
3.055475950241089
3.0353729724884033
3.0163674354553223
2.9983739852905273
2.9813148975372314
2.9651191234588623
2.949721336364746
2.9350602626800537
2.9210798740386963
2.9077296257019043
2.8949618339538574
2.8827340602874756
2.871006965637207
2.8597450256347656
2.8489162921905518
2.838491916656494
2.8284459114074707
2.818753719329834
2.809394359588623
2.8003478050231934
2.791595935821533
2.783123016357422
2.7749135494232178
2.766954183578491
2.75923228263855
2.7517354488372803
2.7444534301757812
2.7373769283294678
2.730494976043701
2.7238008975982666
2.717284917831421
2.7109405994415283
2.7047603130340576
2.698737621307373
2.692866325378418
2.687140703201294
2.6815550327301025
2.67610430717468

In [38]:
# Not much different than before, will be better if we complexify neural net

In [39]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=n_inp).float()
    ctx = itos[ix]
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix_next = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    next_char = itos2[ix_next]
    if ix_next == 0:
        break
    out.append(next_char)
    ctx = (ctx[1], next_char)
    ix = stoi[ctx]
    
  print(''.join(out))

ce
bra
jalius
roczktyharlonimilea
noluwak
